In [1]:
# Connect with the thymio
from tdmclient import ClientAsync
from global_navigation import GlobalNavigation

client = ClientAsync()
node = await client.wait_for_node()
await node.lock()

Node 6ca1ca45-4524-41eb-8cc4-fb023f64b9d0

In [2]:
from tdmclient import ClientAsync
from global_navigation import GlobalNavigation

from Vision import Vision
import time
from motion_control import MotionControl
from robot_api import Thymio
from global_navigation import GlobalNavigation
import numpy as np
from kalman_filter import KalmanFilter

In [3]:
# create vision module object
visio = Vision()
visio.begin()

"""Point d'entrée principal"""
global_nav = GlobalNavigation()
Thymio = Thymio(node, client)
motion_control = MotionControl(Thymio)
kalman_filter =KalmanFilter()

Caméra initialisée sur le port 4


In [4]:
thymio, goal, obstacles = [],[],[]

while not (thymio and goal and obstacles):
    _thymio, _goal, _obstacles = visio.get_thymio_goal_and_obstacles()
    time.sleep(0.2)
    if _thymio : 
        thymio = _thymio
        pre_state = np.array(_thymio)
        kalman_filter.kal_state = np.array(_thymio)
        print(kalman_filter.kal_state)
    if _goal :
        goal = _goal
    if _obstacles :
        obstacles = _obstacles



path = None
# Path planning
while path is None:
    path, _, _ = global_nav.dijkstra(thymio, goal, obstacles)
visio.camera_feed.analysis.path = path.copy()
path.pop(0)
goal_point = path[0]
print("everything ok to start")


[512.7442   230.23267   -2.741352]
everything ok to start


In [1]:
try:
    while True:
        # 0 - global 1 - local
        local_global = motion_control.is_obstacle()
        if local_global == 0:
            # enter the local navigation
            motion_control.obstacle_avoidance()
        if local_global == 1:
            # enter the global navigation 
            if motion_control.path_tracking(kalman_filter.kal_state, goal_point):
                path.pop(0)
                if not path:
                    motion_control.stop_motor()
                    break
                else:
                    goal_point = path[0]
        
        thymio,  *_ = visio.get_thymio_goal_and_obstacles()
        thymio = kalman_filter.check_camera(thymio)
        vl_displacement, vr_displacement = motion_control.get_displacement()
        kal_state, kal_variance = kalman_filter.estimate(vl_displacement, vr_displacement, thymio)
        visio.camera_feed.past_kalman_estimates.append(kal_state[:2])
        if motion_control.is_kidnapping(kal_state, pre_state):
            path, _, _ = global_nav.dijkstra(thymio, goal, obstacles)
            visio.camera_feed.analysis.path = path.copy()
            path.pop(0)
            goal_point = path[0]
        pre_state = kal_state
except KeyboardInterrupt:
    print("Stop the program")
finally:
    visio.stop()
    print("Goal reached")

NameError: name 'visio' is not defined